# 02 · Linkage, cohort and sets

This notebook links every MIMIC-IV-ECG record to its clinical context, applies the exclusions, draws the patient-level splits of MIMIC-IV-ECG and
CODE-15%, and forms the sets of repeat ECGs and the cohorts that the analyses use.

| Output (paths in notebook 00) | Content |
|---|---|
| `ECG_CONTEXT` | per ECG that passed quality control: care setting, admission and emergency department stay, age, sex, death within 1 year |
| `COHORT` | per MIMIC-IV-ECG record: the columns of `MIMIC_RECORDS` and `MIMIC_QC`, the first reason for exclusion, `include` and the split |
| `CODE15_RECORDS` | per row of the CODE-15% arrays: exam, patient, age, sex, `age_valid` and the split |
| `PAIRS_48H`, `PAIRS_1H`, `PAIRS_OUT30` | the three sets of repeat ECGs of the test split |
| `PAIRS_NIGHT_DAY` | pairs of one night and one daytime ECG of the same test patient within 48 hours |
| `MORTALITY_COHORT`, `AVERAGING_7D`, `AVERAGING_48H` | the cohorts of the mortality analysis |
| `COHORT_SUMMARY` | the counts of the flow diagram (Fig. 1b), the splits and the sets |

**Credentialed data.** Everything written here is derived from MIMIC-IV and MIMIC-IV-ED and stays inside `MASTER`. The notebook prints counts only;
counts between 1 and 9 are shown as "<10".

Every output that exists is kept (never overwritten). The whole notebook takes about 1 minute and 3 GB of memory.

In [ ]:
%run ./00_setup_and_settings.ipynb

In [ ]:
import json


def save_new(frame, path):
    """Write a table unless its file exists (an existing file is kept)."""
    if path.exists():
        print(f"  {path.name}: exists, kept")
        return
    frame.to_parquet(path, index=False)
    print(f"  {path.name}: written, {len(frame):,} rows")


def patient_split(ids, seed, shares):
    """Patient-level split: the patients, sorted by identifier, each get one uniform random number from a generator with a fixed seed;
    numbers below the training share go to training, those below training + validation to validation, the rest to test."""
    ids = np.sort(np.asarray(ids))
    u = np.random.default_rng(seed).random(len(ids))
    cut_train, cut_val = np.cumsum([shares["train"], shares["val"]])
    return pd.Series(np.where(u < cut_train, "train", np.where(u < cut_val, "val", "test")), index=ids)

## Quality control

A record is excluded for the first of these reasons that applies (in brackets: the number of records of MIMIC-IV-ECG 1.0 excluded for it):

1. `no_time`: no recording time (0);
2. `hash_fail`: its files differ from the published checksums or could not be read (0);
3. `format_fail`: not 12 leads of 5,000 samples at 500 Hz in the expected order (0);
4. `missing_lead`: a lead with more than 50% missing samples (26);
5. `flat_lead`: a lead whose range (maximum minus minimum) over the whole record is below 0.02 mV (1,950).

In [ ]:
rec = pd.read_parquet(MIMIC_RECORDS)
qc = pd.read_parquet(MIMIC_QC)
coh = rec.merge(qc, on="idx", how="left", validate="one_to_one")
coh["minute"] = coh["minute"].astype("Int16")
print(f"MIMIC-IV-ECG: {len(coh):,} records of {coh.subject_id.nunique():,} patients")

rules = [
    ("no_time", coh["minute"].isna()),
    ("hash_fail", coh["sha_ok"] != 1),
    ("format_fail", coh["format_ok"] != 1),
    ("missing_lead", coh["max_lead_missing_frac"] > 0.5),
    ("flat_lead", coh["n_flat_leads"] > 0),
]
qc_reason = pd.Series("", index=coh.index)
rule_hits = {}
for name, mask in rules:
    mask = mask.fillna(True).astype(bool)                        # unknown values count as a failed check
    rule_hits[name] = int(mask.sum())
    qc_reason = qc_reason.where(~(mask & (qc_reason == "")), name)
qc_pass = (qc_reason == "").to_numpy()
print("records failing each check:", {k: masked(v) for k, v in rule_hits.items()})
print(f"passed quality control: {int(qc_pass.sum()):,} ECGs")

## Clinical context of each ECG

Each ECG that passed quality control is linked by patient and time to MIMIC-IV and MIMIC-IV-ED. For each kind of stay — hospital transfers (each
transfer is a stay in one care unit), stays in intensive care units, and emergency department stays — the stay of the same patient that started last
at or before the ECG is found; the ECG lies inside it if it was recorded before the stay ended. The care setting (`loc`) is then, with the later rule
taking precedence:

1. outpatient;
2. emergency department (ED), if the ECG lies inside an ED stay;
3. the type of the care unit of the hospital transfer, if the ECG lies inside a transfer: ED for transfers of event type "ED" and units named
   "Emergency…"; ICU for units whose name contains "intensive care", "ICU", "(CCU)" or "coronary care"; intermediate ("intermediate", "stepdown",
   "step down"); periop ("PACU", "post anesthesia", "operating", "procedure", "cath"); observation; other ("discharge lounge"); unknown (no unit);
   every other unit is a ward;
4. ICU, if the ECG lies inside a stay in an intensive care unit.

The admission (`hadm_id`) is that of the transfer containing the ECG, and `ed_stay` the ED stay containing it. Chronological age is the anchor age of
the patient plus the year of the ECG minus the anchor year; MIMIC-IV gives no exact age from 89 years, so `age_valid` requires an anchor age and an
age below 89. Death within 1 year (`death_1y`) is a date of death 0–365 days after the date of the ECG, counted in calendar days because the date of
death has no time of day (a death on the day of the ECG counts; one on the 366th day does not).

In [ ]:
def unit_category(careunit):
    """Type of a care unit of MIMIC-IV, from its name."""
    c = str(careunit).lower()
    if "emergency" in c:
        return "ED"
    if "intensive care" in c or "icu" in c or "(ccu)" in c or "coronary care" in c:
        return "ICU"
    if "intermediate" in c or "stepdown" in c or "step down" in c:
        return "intermediate"
    if "pacu" in c or "post anesthesia" in c or "operating" in c or "procedure" in c or "cath" in c:
        return "periop"
    if "observation" in c:
        return "observation"
    if "discharge lounge" in c:
        return "other"
    if c in ("nan", "unknown", ""):
        return "unknown"
    return "ward"


def within(ecg, stays, start, end, keep=()):
    """For each ECG: the stay of the same patient that started last at or before the ECG and whether the ECG lies inside it (before its end).
    Indexed by idx."""
    a = ecg[["idx", "subject_id", "t"]].sort_values("t")
    b = stays.dropna(subset=[start, end]).sort_values(start)
    m = pd.merge_asof(a, b[["subject_id", start, end, *keep]], left_on="t", right_on=start, by="subject_id", direction="backward")
    m["inside"] = (m["t"] < m[end]).fillna(False)
    return m.set_index("idx")


def ecg_context(ecg):
    """Clinical context of the ECGs in `ecg` (idx, subject_id, t). Reads only the columns of MIMIC-IV and MIMIC-IV-ED it needs."""
    tr = pd.read_csv(MIMIC_IV_DIR / "hosp" / "transfers.csv.gz", parse_dates=["intime", "outtime"],
                     usecols=["subject_id", "hadm_id", "eventtype", "careunit", "intime", "outtime"])
    tr["cat"] = tr["careunit"].map(unit_category)
    tr.loc[tr["eventtype"].eq("ED"), "cat"] = "ED"
    icu = pd.read_csv(MIMIC_IV_DIR / "icu" / "icustays.csv.gz", parse_dates=["intime", "outtime"],
                      usecols=["subject_id", "intime", "outtime"])
    eds = pd.read_csv(MIMIC_ED_DIR / "ed" / "edstays.csv.gz", parse_dates=["intime", "outtime"],
                      usecols=["subject_id", "stay_id", "intime", "outtime"])
    pat = pd.read_csv(MIMIC_IV_DIR / "hosp" / "patients.csv.gz", parse_dates=["dod"],
                      usecols=["subject_id", "gender", "anchor_age", "anchor_year", "dod"])
    print(f"MIMIC-IV: {len(tr):,} transfers, {len(icu):,} ICU stays, {len(pat):,} patients; MIMIC-IV-ED: {len(eds):,} ED stays")

    tw = within(ecg, tr.rename(columns={"hadm_id": "tr_hadm"}), "intime", "outtime", keep=("cat", "tr_hadm"))
    iw = within(ecg, icu, "intime", "outtime")
    ew = within(ecg, eds.rename(columns={"stay_id": "ed_stay"}), "intime", "outtime", keep=("ed_stay",))

    e = ecg.set_index("idx")
    ctx = pd.DataFrame(index=e.index)
    loc = pd.Series("outpatient", index=ctx.index)
    loc[ew["inside"].reindex(ctx.index).fillna(False).astype(bool)] = "ED"
    t_in = tw["inside"].reindex(ctx.index).fillna(False).astype(bool)
    loc[t_in] = tw["cat"].reindex(ctx.index)[t_in]
    loc[iw["inside"].reindex(ctx.index).fillna(False).astype(bool)] = "ICU"
    ctx["loc"] = loc
    ctx["hadm_id"] = tw["tr_hadm"].where(t_in).reindex(ctx.index)
    ctx["ed_stay"] = ew["ed_stay"].where(ew["inside"]).reindex(ctx.index)
    ctx["subject_id"] = e["subject_id"]
    t = e["t"]
    p = pat.set_index("subject_id")
    ctx["sex"] = ctx["subject_id"].map(p["gender"])
    anchor_age = ctx["subject_id"].map(p["anchor_age"])
    ctx["age"] = anchor_age + (t.dt.year - ctx["subject_id"].map(p["anchor_year"]))
    ctx["age_valid"] = (anchor_age < 89) & (ctx["age"] < 89) & ctx["age"].notna()
    dod = ctx["subject_id"].map(p["dod"])
    days = (dod - t.dt.normalize()).dt.days                         # calendar days from the date of the ECG to the date of death
    ctx["death_1y"] = (dod.notna() & days.between(0, 365)).astype(int)
    return ctx.reset_index()

## Exclusions by age and the patient-level split of MIMIC-IV-ECG

ECGs that passed quality control are excluded, in this order, when the patient is not in the patients table of MIMIC-IV (`age_unavailable`), when
the age at the ECG or the anchor age of the patient is 89 years or more (`age_89_or_older`; `age_valid` is false), or when the age is below 18 years
(`age_under_18`). MIMIC-IV gives no exact age from 89 years, and the rule on the anchor age also excludes the ECGs that such patients had in years
before the anchor year. The patients with at least one remaining ECG are then split: sorted by `subject_id`, each gets one random number (seed
`MIMIC_SPLIT_SEED`), and numbers below 0.4 go to training, from 0.4 to below 0.5 to validation and the rest to test. All ECGs of a patient carry
the patient's split, also excluded ones; only included ECGs (`include`) are analysed.

In [ ]:
# About half a minute, most of it for reading the MIMIC-IV tables.
if COHORT.exists() and ECG_CONTEXT.exists():
    print("COHORT and ECG_CONTEXT exist, kept")
    coh = pd.read_parquet(COHORT)
    ctx = pd.read_parquet(ECG_CONTEXT)
else:
    ecg = coh.loc[qc_pass, ["idx", "subject_id", "ecg_time"]].copy()
    ecg["t"] = pd.to_datetime(ecg["ecg_time"])
    ctx = ecg_context(ecg)
    by_idx = ctx.set_index("idx")
    age = coh["idx"].map(by_idx["age"])
    valid = coh["idx"].map(by_idx["age_valid"]).eq(True)                    # records without context (failed quality control): False
    age_reason = np.select([age.isna(), ~valid, age < AGE_MIN], ["age_unavailable", "age_89_or_older", "age_under_18"], "")
    coh["exclude_reason"] = np.where(qc_pass, age_reason, qc_reason)
    coh["include"] = coh["exclude_reason"] == ""
    split_of = patient_split(coh.loc[coh["include"], "subject_id"].unique(), MIMIC_SPLIT_SEED, MIMIC_SPLIT)
    coh["split"] = coh["subject_id"].map(split_of)
    save_new(ctx, ECG_CONTEXT)
    save_new(coh, COHORT)

inc = coh[coh["include"]]
print(f"included: {len(inc):,} ECGs of {inc.subject_id.nunique():,} patients")
print("excluded, first reason:", {k: masked(v) for k, v in coh.loc[~coh["include"], "exclude_reason"].value_counts().items()})
for s in ("train", "val", "test"):
    print(f"  {s:5s}: {masked((inc.split == s).sum())} ECGs of {masked(inc.loc[inc.split == s, 'subject_id'].nunique())} patients")
c = ctx[ctx["idx"].isin(inc["idx"])]
print("care setting of the included ECGs (%):", (100 * c["loc"].value_counts(normalize=True)).round(1).to_dict())
print(f"women among the included patients: {100 * (c.drop_duplicates('subject_id')['sex'] == 'F').mean():.1f}%")

## Exclusions and the patient-level split of CODE-15%

The same procedure as for MIMIC-IV-ECG: exams at ages of 18–88 years are kept (`age_valid`); their patients, sorted by `patient_id`, each get one
random number (seed `CODE15_SPLIT_SEED`), below 0.8 training, from 0.8 to below 0.9 validation, the rest test. Exams outside 18–88 years (and the
padding rows of the files, which have no age; notebook 01) get no split.

In [ ]:
if CODE15_RECORDS.exists():
    print("CODE15_RECORDS exists, kept")
    code15 = pd.read_parquet(CODE15_RECORDS)
else:
    code15 = pd.read_parquet(CODE15_EXAMS)
    ok = code15["age"].between(AGE_MIN, AGE_MAX)
    code_split = patient_split(code15.loc[ok, "patient_id"].unique(), CODE15_SPLIT_SEED, CODE15_SPLIT)
    code15["split"] = code15["patient_id"].map(code_split).where(ok)
    code15["age_valid"] = ok
    save_new(code15, CODE15_RECORDS)
print(f"CODE-15%: {len(code15):,} rows, without split {masked(code15.split.isna().sum())} (outside 18-88 years or padding rows)")
for s in ("train", "val", "test"):
    print(f"  {s:5s}: {(code15.split == s).sum():,} exams of {code15.loc[code15.split == s, 'patient_id'].nunique():,} patients")

## Repeat-ECG sets of the test split

All sets are formed from the included ECGs of the test split. A pair consists of two ECGs of the same patient; `idx_a` is the earlier and `idx_b`
the later ECG, and `hours_apart` the time between them.

* `PAIRS_48H`: every pair recorded more than 0 and at most 48 hours apart (the main set);
* `PAIRS_1H`: the pairs of this set recorded at most 1 hour apart, in the same care setting, and during the same admission or the same ED stay or both
  as outpatients;
* `PAIRS_OUT30`: pairs of outpatient ECGs recorded between 08:00 and 17:59 and more than 0 and at most 30 days apart.

ECGs with the same recording time do not form a pair. The pairs are listed in a fixed order: first all pairs of ECGs that are neighbours in their
patient's time order, then the pairs with one ECG between them, and so on. The order does not change any estimate, but the patient bootstrap of the
analyses numbers the patients in the order of the file, so the resamples depend on it.

In [ ]:
ctx_i = ctx.set_index("idx")
test = coh.loc[coh["include"] & (coh["split"] == "test")]
df = pd.DataFrame({"idx": test["idx"].to_numpy(), "subject_id": test["subject_id"].to_numpy(),
                   "t": pd.to_datetime(test["ecg_time"]).to_numpy(), "minute": test["minute"].astype(int).to_numpy()})
for col in ("loc", "hadm_id", "ed_stay", "sex", "death_1y"):
    df[col] = ctx_i.loc[df["idx"], col].to_numpy()
df = df.sort_values(["subject_id", "t", "idx"]).reset_index(drop=True)       # ECGs with the same time: in record order
print(f"test split: {len(df):,} ECGs of {df.subject_id.nunique():,} patients")


def pairs_within(d, max_hours):
    """All pairs (earlier a, later b) of ECGs of the same patient recorded more than 0 and at most max_hours apart; d sorted by patient and time."""
    s, t, idx = d["subject_id"].to_numpy(), d["t"].to_numpy(), d["idx"].to_numpy()
    a, h = [], []
    for k in range(1, len(d)):                                   # k: distance between the two ECGs in the sorted list
        same = s[k:] == s[:-k]
        dt = (t[k:] - t[:-k]) / np.timedelta64(1, "h")
        close = same & (dt <= max_hours)
        if not close.any():                                      # then no pair is further apart in the list either
            break
        r = np.nonzero(close & (dt > 0))[0]
        a.append(np.c_[r, r + k])
        h.append(dt[r])
    a = np.concatenate(a) if a else np.empty((0, 2), np.int64)
    return pd.DataFrame({"idx_a": idx[a[:, 0]], "idx_b": idx[a[:, 1]], "subject_id": s[a[:, 0]],
                         "hours_apart": np.concatenate(h) if h else np.empty(0)})


def attach(p, d, cols):
    """Columns of the earlier (_a) and later (_b) ECG of each pair."""
    di = d.set_index("idx")
    for s in ("a", "b"):
        for col in cols:
            p[f"{col}_{s}"] = di.loc[p[f"idx_{s}"], col].to_numpy()
    return p


p48 = pairs_within(df, WITHIN_48H)
q = attach(p48.copy(), df, ["loc", "hadm_id", "ed_stay"])
same_visit = ((q.hadm_id_a == q.hadm_id_b) & q.hadm_id_a.notna()) | ((q.ed_stay_a == q.ed_stay_b) & q.ed_stay_a.notna()) | \
             ((q.loc_a == "outpatient") & (q.loc_b == "outpatient"))
p1 = p48[((q.hours_apart <= WITHIN_1H) & (q.loc_a == q.loc_b) & same_visit).to_numpy()].reset_index(drop=True)
outpatient = df[(df["loc"] == "outpatient") & (df["minute"] // 60).between(*OUTPATIENT_HOURS)]
p30 = pairs_within(outpatient, OUTPATIENT_DAYS * 24.0)
for name, p, path in (("within 48 h", p48, PAIRS_48H), ("within 1 h, same visit", p1, PAIRS_1H), ("outpatient, 30 days", p30, PAIRS_OUT30)):
    print(f"{name}: {masked(p.subject_id.nunique())} patients, {masked(len(p))} pairs")
    save_new(p, path)

## Night–daytime pairs

For the supplementary comparison of night and daytime ECGs: every pair of an ECG recorded at night (00:00–05:59, `idx_a`) and one recorded in the
daytime (10:00–15:59, `idx_b`) of the same test patient, at most 48 hours apart in either order (`hours_apart` is the absolute time between them).
The pairs are listed by patient and, within a patient, by the time of the night ECG and then of the daytime ECG.

In [ ]:
def night_day_pairs(d, max_hours):
    """Every pair of a night ECG (idx_a) and a daytime ECG (idx_b) of the same patient at most max_hours apart; d sorted by patient and time."""
    hour = d["minute"] // 60
    night, day = hour.between(*NIGHT_HOURS), hour.between(*DAYTIME_HOURS)
    out = []
    for _, g in d[night | day].groupby("subject_id", sort=False):
        A, B = g[night[g.index]], g[day[g.index]]
        if len(A) and len(B):
            dt = np.abs((A["t"].to_numpy()[:, None] - B["t"].to_numpy()[None, :]) / np.timedelta64(1, "h"))
            ia, ib = np.nonzero((dt > 0) & (dt <= max_hours))
            if len(ia):
                out.append(pd.DataFrame({"idx_a": A["idx"].to_numpy()[ia], "idx_b": B["idx"].to_numpy()[ib],
                                         "subject_id": A["subject_id"].to_numpy()[ia], "hours_apart": dt[ia, ib]}))
    return pd.concat(out, ignore_index=True)


pnd = night_day_pairs(df, WITHIN_48H)
print(f"night-daytime pairs: {masked(pnd.subject_id.nunique())} patients, {masked(len(pnd))} pairs")
save_new(pnd, PAIRS_NIGHT_DAY)

## Cohorts of the mortality analysis

From the included ECGs of the test patients with known sex, sorted by patient and time:

* `MORTALITY_COHORT`: per patient, the first ECG that is followed by another ECG more than 0 and at most 48 hours later (`idx_first`) and that next
  ECG (`idx_next`);
* `AVERAGING_7D`, `AVERAGING_48H`: per patient, the first run of five consecutive ECGs, each recorded after the previous one, with the fifth at most
  7 days (168 hours) or 48 hours after the first (`idx_1` … `idx_5`).

`n_ecgs` is the number of ECGs of the patient from the first ECG of the cohort to 48 hours (mortality cohort, averaging within 48 hours) or 7 days
(averaging within 7 days) later, the first included, by which the mortality models are adjusted (notebook 06). ECGs recorded at the same time as the
first ECG but listed before it are not counted.

In [ ]:
def first_runs(subj, t, k, hours):
    """Row of the first ECG of each patient that starts k consecutive ECGs (each strictly later than the previous) spanning at most `hours`;
    subj and t sorted by patient and time."""
    m = len(subj) - k + 1
    ok = (subj[k - 1:] == subj[:m]) & ((t[k - 1:] - t[:m]) / np.timedelta64(1, "h") <= hours)
    for j in range(1, k):
        ok &= t[j:j + m] > t[j - 1:j - 1 + m]
    i = np.nonzero(ok)[0]
    return pd.Series(i).groupby(subj[i]).first().to_numpy()


def ecgs_within(subj, t, rows, hours):
    """Number of ECGs of the same patient from each of `rows` up to `hours` later (the row itself included)."""
    stops = np.r_[np.nonzero(subj[1:] != subj[:-1])[0] + 1, len(subj)]
    end_of = np.repeat(stops, np.diff(np.r_[0, stops]))              # end of the rows of each row's patient
    lim = np.timedelta64(int(hours * 3600), "s")
    return np.array([np.searchsorted(t[r:end_of[r]], t[r] + lim, side="right") for r in rows], dtype=np.int64)


known = df[df["sex"].notna()]
subj, tt, rec_idx = known["subject_id"].to_numpy(), known["t"].to_numpy(), known["idx"].to_numpy()
i2 = first_runs(subj, tt, 2, WITHIN_48H)
mortality = pd.DataFrame({"idx_first": rec_idx[i2], "idx_next": rec_idx[i2 + 1], "subject_id": subj[i2],
                          "n_ecgs": ecgs_within(subj, tt, i2, WITHIN_48H)})
print(f"mortality cohort: {masked(len(mortality))} patients, {masked(known.loc[known.idx.isin(mortality.idx_first), 'death_1y'].sum())} "
      f"deaths within 1 year")
save_new(mortality, MORTALITY_COHORT)
averaging = {}
for name, hours, path in (("within 7 days", WITHIN_7D, AVERAGING_7D), ("within 48 h", WITHIN_48H, AVERAGING_48H)):
    i5 = first_runs(subj, tt, 5, hours)
    averaging[name] = pd.DataFrame({**{f"idx_{k + 1}": rec_idx[i5 + k] for k in range(5)}, "subject_id": subj[i5],
                                    "n_ecgs": ecgs_within(subj, tt, i5, hours)})
    print(f"averaging, five ECGs {name}: {masked(len(i5))} patients")
    save_new(averaging[name], path)

## Counts of the flow diagram and of the splits

`COHORT_SUMMARY` holds the counts of Fig. 1b (records, exclusions, included ECGs and patients, splits), the CODE-15% split and the sizes of the sets.

In [ ]:
excl = coh.loc[~coh["include"], "exclude_reason"].value_counts()
first_reason = {k: int(v) for k, v in excl.items() if k not in ("age_89_or_older", "age_unavailable", "age_under_18")}
first_reason.update({k: int(excl.get(k, 0)) for k in ("age_89_or_older", "age_unavailable", "age_under_18")})
summary = {
    "records": int(len(coh)),
    "included": int(coh["include"].sum()),
    "excluded_by_first_reason": first_reason,
    "patients_included": int(inc.subject_id.nunique()),
    "mimic": {"seed": MIMIC_SPLIT_SEED, "cuts": [float(x) for x in np.cumsum([MIMIC_SPLIT["train"], MIMIC_SPLIT["val"]])]},
    "split_records": {k: int(v) for k, v in inc["split"].value_counts().items()},
    "split_subjects": {k: int(v) for k, v in inc.groupby("split")["subject_id"].nunique().items()},
    "code15": {"seed": CODE15_SPLIT_SEED, "cuts": [float(x) for x in np.cumsum([CODE15_SPLIT["train"], CODE15_SPLIT["val"]])],
               "exams": int(len(code15)), "excluded_age_outside_18_88": int((~code15["age_valid"]).sum()),
               "padding_rows": int((code15["exam_id"] == 0).sum()),
               "split_exams": {k: int(v) for k, v in code15["split"].value_counts().items()},
               "split_patients": {k: int(v) for k, v in code15.dropna(subset=["split"]).groupby("split")["patient_id"].nunique().items()}},
    "sets": {name: {"patients": int(p.subject_id.nunique()), "pairs": int(len(p))}
             for name, p in (("within_48h", p48), ("within_1h", p1), ("outpatient_30d", p30), ("night_daytime", pnd))},
    "cohorts": {"mortality": int(len(mortality)), "averaging_within_7d": int(len(averaging["within 7 days"])),
                "averaging_within_48h": int(len(averaging["within 48 h"]))},
}
if COHORT_SUMMARY.exists():
    print("COHORT_SUMMARY exists, kept")
else:
    COHORT_SUMMARY.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("COHORT_SUMMARY written")
excluded_qc = int((~coh["include"] & coh["exclude_reason"].isin([name for name, _ in rules])).sum())
print(f"Fig. 1b: {summary['records']:,} records; excluded: quality control (flat or missing lead) {excluded_qc:,}, age outside 18-88 years or "
      f"unknown {int((~coh['include']).sum()) - excluded_qc:,}; included {summary['included']:,} ECGs of {summary['patients_included']:,} patients")